# Statistical models for intensity-change forecasting

Predict $\Delta V_{24} = V_{\max}(t+24\,\mathrm{h}) - V_{\max}(t)$ (kt) from the
cyclone state and environment at issue time $t$.

In [31]:
import json
import sys
import time
from itertools import combinations
from pathlib import Path

import numpy as np
import pandas as pd
import sklearn
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots
from scipy.stats import loguniform, randint, uniform

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import ConstantKernel as C, RBF, Matern, WhiteKernel
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.model_selection import GroupKFold, RandomizedSearchCV
from sklearn.neighbors import KNeighborsRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.svm import SVR

# xgboost before torch/lightgbm so the OpenMP runtimes load in a safe order
# (macOS segfaults otherwise - see the guided notebook).
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor

pio.templates.default = "plotly_white"

SEED = 20260909 
np.random.seed(SEED)

ORANGE, PURPLE = "#E66100", "#5D3A9B"   # baselines vs models
MODEL_COLORS = ["#E66100", "#5D3A9B", "#648FFF", "#DC267F", "#31C9B0", "#FFB000"]

print(f"python {sys.version.split()[0]} | numpy {np.__version__} | "
      f"pandas {pd.__version__} | scikit-learn {sklearn.__version__}")

python 3.13.0 | numpy 2.5.2 | pandas 3.0.5 | scikit-learn 1.9.0


## 1. Load the data

In [32]:
REPO_ROOT = next(parent for parent in [Path.cwd(), *Path.cwd().parents]
                 if (parent / "tuesday" / "outputs").is_dir())
TUESDAY_OUTPUTS = REPO_ROOT / "tuesday" / "outputs"
OUTPUT_DIRECTORY = REPO_ROOT / "wednesday" / "outputs"
MODEL_DIRECTORY = REPO_ROOT / "wednesday" / "models"
for directory in (OUTPUT_DIRECTORY, MODEL_DIRECTORY):
    directory.mkdir(parents=True, exist_ok=True)

# Global lead time.
LEAD_HOURS = 24
assert LEAD_HOURS in (6, 12, 24, 48, 72)
TARGET = f"delta_v_{LEAD_HOURS}"
FUTURE_WIND = f"wind_tplus_{LEAD_HOURS}"

dataset_path = TUESDAY_OUTPUTS / "eumacc_guided_ml_ready.csv.gz"
data = pd.read_csv(dataset_path, keep_default_na=False, na_values=["", " "],
                   parse_dates=["ISO_TIME"], low_memory=False)

manifest_path = TUESDAY_OUTPUTS / "eumacc_guided_ml_ready_manifest.json"
manifest = json.loads(manifest_path.read_text())
assert len(data) == manifest["rows"]
assert data["SID"].nunique() == manifest["storms"]
for split_name, expected in manifest["split_rows"].items():
    assert int(data["split"].eq(split_name).sum()) == expected, split_name
assert not data.duplicated(["SID", "ISO_TIME"]).any()

In [33]:
def partition_violations(frame, identifier, partition="split"):
    """Identifier values seen under more than one fold, as {(foldA, foldB): ids}."""
    assert not frame[partition].isna().any()
    assert not frame[identifier].isna().any()
    pairs = frame[[identifier, partition]].drop_duplicates()
    straddling = pairs.loc[pairs.duplicated(identifier, keep=False)]
    seen_in = {fold: set(ids) for fold, ids in
               straddling.groupby(partition, sort=False)[identifier]}
    violations = {}
    folds = frame[partition].unique()
    for first, second in combinations(folds, 2):
        shared = seen_in.get(first, set()) & seen_in.get(second, set())
        if shared:
            violations[(first, second)] = sorted(shared)
    return violations


def assert_partition_intact(frame, identifier, partition="split"):
    violations = partition_violations(frame, identifier, partition)
    if violations:
        report = "; ".join(f"{len(ids)} shared between {a} and {b}"
                           for (a, b), ids in violations.items())
        raise AssertionError(f"'{identifier}' breaches the '{partition}' partition: {report}")


assert_partition_intact(data, "SID")

# Drop the test period once, here. Everything downstream works on modelling_data.
modelling_data = data.loc[data["split"].isin(["train", "validation"])].copy()
assert not modelling_data["SEASON"].ge(2021).any()
del data

display(modelling_data.groupby("split", sort=False).agg(
    rows=("SID", "size"), storms=("SID", "nunique"),
    target_available=(TARGET, lambda c: c.notna().sum())))

,rows,storms,target_available
split,,,
train,22799,1089,22799
validation,5813,289,5813


## 2. Features and the leakage guard

* `STORM_DIR` enters as $(\sin\theta, \cos\theta)$ rather than raw degrees
* `r50_mean_nm` / `r64_mean_nm` are dropped because they are unreported most of the time on the training fold.

The `FORBIDDEN` set names everything a model must never see - future measurements, identifiers, and `SEASON`/`split`, since the season is the split.

In [34]:
modelling_data["storm_dir_sin"] = np.sin(np.deg2rad(modelling_data["STORM_DIR"]))
modelling_data["storm_dir_cos"] = np.cos(np.deg2rad(modelling_data["STORM_DIR"]))

HISTORY = [
    "USA_WIND", "USA_PRES",
    "wind_lag_6", "wind_lag_12", "wind_lag_24",
    "delta_v_past_6", "delta_v_past_12", "delta_v_past_24",
]
RADII = ["USA_RMW", "r34_mean_nm", "r50_mean_nm", "r64_mean_nm"]
KINEMATICS = ["LAT", "LON", "STORM_SPEED", "storm_dir_sin", "storm_dir_cos",
              "DIST2LAND", *RADII]
ENVIRONMENT = [
    "era5_shear_850_200_r200_500_ms", "era5_shear_850_500_r200_500_ms",
    "era5_shear_u_850_200_r200_500_ms", "era5_shear_v_850_200_r200_500_ms",
    "era5_rh_850_700_r200_500_pct", "era5_rh_700_500_r200_500_pct",
    "era5_rh_500_300_r200_500_pct",
    "era5_sst_r0_200_mean_c", "era5_sst_r0_500_mean_c", "era5_sst_r0_500_std_c",
    "era5_sst_ahead_minus_behind_r0_500_c", "era5_mslp_r200_500_mean_hpa",
    "era5_pi_vmax_kt", "pi_headroom_kt",
    "era5_theta_500_minus_850_r200_500_k",
]
CATEGORICAL = ["BASIN"]
BLOCKS = {"history": HISTORY, "kinematics": KINEMATICS,
          "environment": ENVIRONMENT, "categorical": CATEGORICAL}

NUMERIC_FEATURES = HISTORY + KINEMATICS + ENVIRONMENT
ALL_FEATURES = NUMERIC_FEATURES + CATEGORICAL
GROUP = "SID"

EXCLUDED_COLUMNS = ["r50_mean_nm", "r64_mean_nm"]
ACCEPTED_NUMERIC = [c for c in NUMERIC_FEATURES if c not in EXCLUDED_COLUMNS]
ACCEPTED_FEATURES = ACCEPTED_NUMERIC + CATEGORICAL

FORBIDDEN = set()
FORBIDDEN |= {c for c in modelling_data.columns
              if c.startswith(("wind_tplus_", "max_wind_within_", "event_ge"))}
FORBIDDEN |= {c for c in modelling_data.columns
              if c.startswith("delta_v_") and not c.startswith("delta_v_past_")}
FORBIDDEN |= {"SID", "NAME", "ISO_TIME", "USA_IFLAG", "SEASON", "split"}


def assert_no_leakage(columns, name="feature list"):
    leaked = sorted(FORBIDDEN.intersection(columns))
    assert not leaked, f"future or identifying information in {name}: {leaked}"
    return list(columns)


assert_no_leakage(ACCEPTED_FEATURES, "ACCEPTED_FEATURES")
print(f"{len(FORBIDDEN)} columns off-limits; "
      f"{len(ACCEPTED_NUMERIC)} numeric + {len(CATEGORICAL)} categorical kept")

31 columns off-limits; 31 numeric + 1 categorical kept


In [35]:
train = modelling_data.loc[modelling_data["split"].eq("train")].copy()
validation = modelling_data.loc[modelling_data["split"].eq("validation")].copy()
X_train, y_train = train[ACCEPTED_FEATURES], train[TARGET]
X_validation, y_validation = validation[ACCEPTED_FEATURES], validation[TARGET]
has_target_train = y_train.notna()

# +-inf would poison StandardScaler and SVR; turn it into NaN so the imputer sees it.
n_inf = int(np.isinf(modelling_data[ACCEPTED_NUMERIC].to_numpy(dtype=float)).sum())
if n_inf:
    print(f"replacing {n_inf} +-inf values with NaN")
for frame in (modelling_data, train, validation):
    frame[ACCEPTED_NUMERIC] = frame[ACCEPTED_NUMERIC].replace([np.inf, -np.inf], np.nan)

print(f"train {int(has_target_train.sum()):,} cases – "
      f"{train.loc[has_target_train, GROUP].nunique()} storms | "
      f"validation {y_validation.notna().sum():,} cases – "
      f"{validation.loc[y_validation.notna(), GROUP].nunique()} storms")

# Counting every look at the validation set; written out with the run record.
VALIDATION_DECISIONS = []

def note_validation_decision(description, candidates=1, section=""):
    VALIDATION_DECISIONS.append({"section": section, "decision": description,
                                 "candidates": int(candidates)})
    total = sum(d["candidates"] for d in VALIDATION_DECISIONS)
    print(f"[ledger] +{candidates}: {description} (total {total})")

train 22,799 cases – 1089 storms | validation 5,813 cases – 289 storms


## 3. Evaluation protocol

MAE, RMSE, bias, and RMSE skill against persistence
($1 - \mathrm{RMSE}_{model}/\mathrm{RMSE}_{persistence}$; positive beats persistence).

Validation cases share storms, so a row bootstrap would be far too narrow. All
intervals and paired tests below resample **storms** with rows kept together. Two
models whose 95% storm-block intervals overlap count as tied.

In [36]:
def evaluate(y_true, y_predicted, reference_predicted=None):
    """Scores relative to a reference forecast; no reference means persistence."""
    y_true = np.asarray(y_true, dtype=float)
    y_predicted = np.asarray(y_predicted, dtype=float)
    if reference_predicted is None:
        reference_predicted = np.zeros_like(y_true)
    reference_predicted = np.asarray(reference_predicted, dtype=float)
    model_rmse = float(np.sqrt(mean_squared_error(y_true, y_predicted)))
    reference_rmse = float(np.sqrt(mean_squared_error(y_true, reference_predicted)))
    return {
        "mae": float(mean_absolute_error(y_true, y_predicted)),
        "rmse": model_rmse,
        "bias": float(np.mean(y_predicted - y_true)),
        "skill": float(1.0 - model_rmse / reference_rmse),
    }


def storm_block_resample(frame, rng, group=GROUP):
    """Row positions of one bootstrap draw, sampling storms with replacement."""
    storm_ids = frame[group].to_numpy()
    unique_storms = np.unique(storm_ids)
    rows_of = {s: np.flatnonzero(storm_ids == s) for s in unique_storms}
    drawn = rng.choice(unique_storms, size=unique_storms.size, replace=True)
    return np.concatenate([rows_of[s] for s in drawn])


def storm_block_mae_interval(frame, y_predicted, n_resamples=400, seed=SEED):
    """95% interval for MAE over storm resamples."""
    rng = np.random.default_rng(seed)
    errors = np.abs(frame[TARGET].to_numpy(float) - np.asarray(y_predicted, float))
    resampled = np.array([errors[storm_block_resample(frame, rng)].mean()
                          for _ in range(n_resamples)])
    return float(np.percentile(resampled, 2.5)), float(np.percentile(resampled, 97.5))


def paired_storm_block_deltas(frame, y_predicted, y_reference, n_resamples=500, seed=SEED):
    """Paired bootstrap of MAE(model) - MAE(reference) over the same storm draws.

    Stricter than comparing two marginal intervals. An interval containing 0 = tied.
    """
    rng = np.random.default_rng(seed)
    truth = frame[TARGET].to_numpy(float)
    errors = np.abs(truth - np.asarray(y_predicted, float))
    reference_errors = np.abs(truth - np.asarray(y_reference, float))
    deltas = np.array([errors[draw].mean() - reference_errors[draw].mean()
                       for draw in (storm_block_resample(frame, rng)
                                    for _ in range(n_resamples))])
    return (float(deltas.mean()), float(np.percentile(deltas, 2.5)),
            float(np.percentile(deltas, 97.5)))


def split_frame(split="validation"):
    """Scored rows of one fold. 'test' stays blocked until Thursday's notebook."""
    assert split in ("train", "validation"), f"'{split}' is sealed"
    frame = modelling_data.loc[modelling_data["split"].eq(split)].copy()
    return frame.loc[frame[TARGET].notna()].copy()


def score_all_models(results_frame, model_names):
    """One metrics row per model, each against persistence on the same rows."""
    persistence_mae = float(results_frame[TARGET].abs().mean())
    records = []
    for model_name in model_names:
        predicted = results_frame[model_name].to_numpy(float)
        row = evaluate(results_frame[TARGET], predicted)
        row["mae_low"], row["mae_high"] = storm_block_mae_interval(results_frame, predicted)
        row.update(persistence_mae=persistence_mae, cases=len(results_frame),
                   storms=results_frame[GROUP].nunique())
        records.append({"model": model_name, **row})
    return pd.DataFrame(records).set_index("model")


def rank_by_interval(frame):
    """Walk down the MAE table; models whose intervals overlap share a rank."""
    ordered = frame.sort_values("mae")
    ranks, current_rank, group_high = [], 1, ordered.iloc[0]["mae_high"]
    for _, row in ordered.iterrows():
        if row["mae_low"] > group_high:
            current_rank += 1
            group_high = row["mae_high"]
        else:
            group_high = min(group_high, row["mae_high"])
        ranks.append(current_rank)
    scored = ordered.copy()
    scored["rank"] = ranks
    return scored


def plot_leaderboard(frame, title):
    """Horizontal MAE bars with the 95% storm-block interval as an error bar."""
    ordered = frame.sort_values("mae")
    is_baseline = np.asarray(ordered.index.str.contains("persistence|climatology|rolling"))
    figure = go.Figure()
    figure.add_bar(
        x=ordered["mae"], y=list(ordered.index), orientation="h",
        error_x=dict(type="data",
                     array=ordered["mae_high"] - ordered["mae"],
                     arrayminus=ordered["mae"] - ordered["mae_low"],
                     color="0.25", thickness=1.2),
        marker_color=np.where(is_baseline, ORANGE, PURPLE),
        hovertemplate="%{y}<br>MAE %{x:.3f} kt<extra></extra>",
    )
    figure.update_layout(
        title=title, showlegend=False, height=0.34 * len(ordered) * 30 + 160,
        xaxis_title="validation MAE (kt), with 95% storm-block interval")
    figure.show()

print("evaluation helpers ready")

evaluation helpers ready


## 4. Rule-based baselines

Persistence (no change), a rolling average of the last past 6-hourly changes, and
two climatologies: the plain training-mean change, and one conditioned on
basin x intensity band. The latter is the bar any real model has to clear.

In [ ]:
ROLLING_AVERAGE_STEPS = 3
PAST_CHANGE_COLUMNS = ["delta_v_past_6", "delta_v_past_12", "delta_v_past_24"]
assert 1 <= ROLLING_AVERAGE_STEPS <= 3

RULE_BASED_BASELINES = ("persistence", "rolling_average", "climatology",
                        "climatology by basin x intensity")

scoring_frame = split_frame("validation")
predictions = pd.DataFrame(index=scoring_frame.index)
predictions["persistence"] = 0.0
# track starts have missing past lags -> fall back to persistence (0.0 change)
predictions["rolling_average"] = scoring_frame[
    PAST_CHANGE_COLUMNS[:ROLLING_AVERAGE_STEPS]].mean(axis=1).fillna(0.0)

training_mean_change = float(y_train.mean())
predictions["climatology"] = training_mean_change

intensity_bins = [0, 34, 64, 96, 300]
intensity_labels = ["tropical storm", "cat 1-2", "cat 3", "cat 4-5"]


def intensity_band(wind):
    return pd.cut(wind, bins=intensity_bins, labels=intensity_labels)


conditional_means = (
    train.assign(band=intensity_band(train["USA_WIND"]))
    .groupby(["BASIN", "band"], observed=True)[TARGET].mean())
lookup_key = pd.MultiIndex.from_arrays(
    [scoring_frame["BASIN"], intensity_band(scoring_frame["USA_WIND"])])
conditional_prediction = conditional_means.reindex(lookup_key).to_numpy()
# unseen basin/band combinations fall back to the global mean
predictions["climatology by basin x intensity"] = np.where(
    np.isnan(conditional_prediction), training_mean_change, conditional_prediction)

assert np.isfinite(predictions.to_numpy(dtype=float)).all(), "NaN in baseline predictions"
print(f"baselines on {len(scoring_frame):,} cases / {scoring_frame[GROUP].nunique()} storms")
note_validation_decision("four rule-based baselines scored and compared",
                         len(RULE_BASED_BASELINES), section="4")

## 5. Model definitions

Ten families, two hand-picked configurations each. The preprocessing follows the
guided notebook's per-family decisions:

- **scaled + imputed** (linear models, SVR, MLP, kNN, the Gaussian process need
  comparable units and complete input): median imputation, standardisation,
  one-hot `BASIN`;
- **imputed, unscaled** for the random forest (trees care about ordering, not units);
- **raw input** for XGBoost, LightGBM, CatBoost and the histogram gradient booster,
  which all handle NaN natively - useful here because a missing lag marks the
  start of a track, and imputing erases that.

Two special cases:

- The **Gaussian process** is fitted on a fixed 2,000-row subsample of the
  training fold - an exact GP is cubic in the number of rows, so 30,000 is not an
  option. Its numbers are honest but its data budget is not comparable to the
  other models; keep that in mind when ranking it.
- The **logistic regression** is a *directional* model, not an intensity model: it
  classifies the sign of $\Delta V_h$ and maps the class to $\pm$ the training mean
  absolute change. Its magnitude is constant by construction, so read its MAE and
  ignore its RMSE.

In [ ]:
ESTIMATOR_CLASSES = {
    "logistic_regression": LogisticRegression,
    "svr": SVR,
    "gaussian_process": GaussianProcessRegressor,
    "random_forest": RandomForestRegressor,
    "hist_gradient_boosting": HistGradientBoostingRegressor,
    "xgboost": XGBRegressor,
    "lightgbm": LGBMRegressor,
    "catboost": CatBoostRegressor,
    "mlp": MLPRegressor,
    "knn": KNeighborsRegressor,
}

# (needs scaling, needs imputation)
PREPROCESSING_NEEDS = {
    "logistic_regression": (True, True),
    "svr": (True, True),
    "gaussian_process": (True, True),
    "random_forest": (False, True),
    "hist_gradient_boosting": (False, False),
    "xgboost": (False, False),
    "lightgbm": (False, False),
    "catboost": (False, False),
    "mlp": (True, True),
    "knn": (True, True),
}

FIXED_KWARGS = {
    "logistic_regression": {"max_iter": 2000, "class_weight": "balanced",
                            "random_state": SEED},
    "svr": {},
    "gaussian_process": {"normalize_y": True, "random_state": SEED},
    "random_forest": {"n_jobs": 1, "random_state": SEED},
    "hist_gradient_boosting": {"random_state": SEED},
    "xgboost": {"n_jobs": 1, "random_state": SEED, "tree_method": "hist"},
    "lightgbm": {"n_jobs": 1, "random_state": SEED, "verbose": -1},
    "catboost": {"random_seed": SEED, "verbose": 0, "thread_count": 1,
                 "allow_writing_files": False},
    "mlp": {"activation": "relu", "max_iter": 500, "early_stopping": False,
            "random_state": SEED},
    "knn": {"n_jobs": 1},
}

MODEL_VARIANTS = {
    "logistic_regression": [
        {"C": 1.0},
        {"C": 0.01},
    ],
    "svr": [
        {"kernel": "rbf", "C": 1.0, "epsilon": 5.0},
        {"kernel": "rbf", "C": 10.0, "epsilon": 2.0},
    ],
    "gaussian_process": [
        {"kernel": C(1.0) * RBF(length_scale=1.0) + WhiteKernel(noise_level=1.0)},
        {"kernel": C(1.0) * Matern(length_scale=1.0, nu=1.5) + WhiteKernel(noise_level=1.0)},
    ],
    "random_forest": [
        {"n_estimators": 300, "min_samples_leaf": 5},
        {"n_estimators": 600, "min_samples_leaf": 1, "max_features": 0.5},
    ],
    "hist_gradient_boosting": [
        {"max_iter": 400, "learning_rate": 0.05},
        {"max_iter": 800, "learning_rate": 0.02, "max_leaf_nodes": 31},
    ],
    "xgboost": [
        {"n_estimators": 400, "learning_rate": 0.05, "max_depth": 6,
         "subsample": 0.8, "colsample_bytree": 0.8},
        {"n_estimators": 800, "learning_rate": 0.02, "max_depth": 4,
         "subsample": 0.8, "colsample_bytree": 0.8},
    ],
    "lightgbm": [
        {"n_estimators": 400, "learning_rate": 0.05, "num_leaves": 63,
         "subsample": 0.8, "colsample_bytree": 0.8},
        {"n_estimators": 800, "learning_rate": 0.02, "num_leaves": 31,
         "subsample": 0.8, "colsample_bytree": 0.8},
    ],
    "catboost": [
        {"iterations": 400, "learning_rate": 0.05, "depth": 6},
        {"iterations": 800, "learning_rate": 0.02, "depth": 4},
    ],
    "mlp": [
        {"hidden_layer_sizes": (128, 64), "alpha": 1e-3, "learning_rate_init": 1e-3},
        {"hidden_layer_sizes": (64, 32), "alpha": 1e-2, "learning_rate_init": 5e-4},
    ],
    "knn": [
        {"n_neighbors": 15, "weights": "distance"},
        {"n_neighbors": 50, "weights": "uniform"},
    ],
}


def build_preprocessor(numeric_columns, categorical_features=(), impute=True, scale=True):
    steps = []
    if impute:
        steps.append(("impute", SimpleImputer(strategy="median")))
    if scale:
        steps.append(("scale", StandardScaler()))
    numeric = Pipeline(steps) if steps else "passthrough"
    categorical = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore", drop="first", sparse_output=False)),
    ])
    return ColumnTransformer([
        ("numeric", numeric, list(numeric_columns)),
        ("categorical", categorical, list(categorical_features)),
    ])


def make_model(architecture, **kwargs):
    """Fresh pipeline: this family's preprocessor + estimator with these settings."""
    scale, impute = PREPROCESSING_NEEDS[architecture]
    estimator = ESTIMATOR_CLASSES[architecture](
        **{**FIXED_KWARGS.get(architecture, {}), **kwargs})
    return Pipeline([
        ("prepare", build_preprocessor(ACCEPTED_NUMERIC, CATEGORICAL,
                                       impute=impute, scale=scale)),
        ("model", estimator),
    ])


model_definitions = {}
variant_records = []
for architecture, variants in MODEL_VARIANTS.items():
    for variant_number, kwargs in enumerate(variants, start=1):
        model_name = f"{architecture}_v{variant_number}"
        model_definitions[model_name] = make_model(architecture, **kwargs)
        variant_records.append({
            "model": model_name, "architecture": architecture,
            "hyperparameters": ", ".join(f"{k}={v}" for k, v in kwargs.items()),
        })

display(pd.DataFrame(variant_records))

,model,architecture,hyperparameters
0,logistic_regression_v1,logistic_regression,C=1.0
1,logistic_regression_v2,logistic_regression,C=0.01
2,svr_v1,svr,"kernel=rbf, C=1.0, epsilon=5.0"
3,svr_v2,svr,"kernel=rbf, C=10.0, epsilon=2.0"
4,gaussian_process_v1,gaussian_process,kernel=1**2 * RBF(length_scale=1) + WhiteKerne...
5,gaussian_process_v2,gaussian_process,"kernel=1**2 * Matern(length_scale=1, nu=1.5) +..."
6,random_forest_v1,random_forest,"n_estimators=300, min_samples_leaf=5"
7,random_forest_v2,random_forest,"n_estimators=600, min_samples_leaf=1, max_feat..."
8,hist_gradient_boosting_v1,hist_gradient_boosting,"max_iter=400, learning_rate=0.05"
9,hist_gradient_boosting_v2,hist_gradient_boosting,"max_iter=800, learning_rate=0.02, max_leaf_nod..."


## 6. Fit the variants

All hyperparameters here were fixed before looking at any validation score. The
directional model is fitted on the sign of the training target, the Gaussian
process on its subsample, everything else on the rows that have a target.

In [ ]:
GP_TRAIN_ROWS = 2000
gp_train = train.loc[has_target_train].sample(GP_TRAIN_ROWS, random_state=SEED)

fitted_pipelines = {}
for model_name, pipeline in model_definitions.items():
    started = time.perf_counter()
    if model_name.startswith("logistic_regression"):
        pipeline.fit(X_train, (y_train > 0).astype(int))
    elif model_name.startswith("gaussian_process"):
        pipeline.fit(gp_train[ACCEPTED_FEATURES], gp_train[TARGET])
    else:
        pipeline.fit(X_train.loc[has_target_train], y_train.loc[has_target_train])
    fitted_pipelines[model_name] = pipeline

    predicted = pipeline.predict(validation[ACCEPTED_FEATURES])
    if model_name.startswith("logistic_regression"):
        predicted = np.where(predicted == 1, 1.0, -1.0) * float(y_train.abs().mean())
    assert np.isfinite(predicted).all(), model_name
    predictions[model_name] = predicted
    print(f"fitted {model_name:32s} in {time.perf_counter() - started:5.1f} s")

note_validation_decision("estimator variants scored on validation",
                         len(model_definitions), section="6")

fitted logistic_regression_v1           in   1.5 s
fitted logistic_regression_v2           in   0.9 s
fitted svr_v1                           in   9.9 s
fitted svr_v2                           in  15.3 s
fitted gaussian_process_v1              in   9.4 s
fitted gaussian_process_v2              in  10.0 s
fitted random_forest_v1                 in  67.6 s


## 7. Tuned models

Each family except the Gaussian process also gets a random search (the exact GP's
cubic cost makes a search over it a bad deal for the information it returns).
Protocol:

- inner CV is `GroupKFold` on `SID` - shuffled folds would score near-duplicates of
  the training rows and flatter the search;
- the search runs inside the training split only; validation is scored once per
  winner, at the end;
- the final model of the notebook is chosen among these winners plus the untuned
  variants, on validation MAE.

In [ ]:
N_SEARCH_ITERATIONS = 2    # candidates per architecture
CV_FOLDS = 2

# ARCHITECTURES_TO_TUNE = ["svr", "random_forest", "hist_gradient_boosting", "xgboost", "lightgbm", "catboost", "mlp", "knn"]
ARCHITECTURES_TO_TUNE = ["svr", "xgboost", "lightgbm", "catboost"]


PARAM_DISTRIBUTIONS = {
    "svr": {
        "model__C": loguniform(1e-1, 1e2),
        "model__epsilon": loguniform(5e-1, 1e1),
        "model__gamma": ["scale", "auto"],
    },
    "random_forest": {
        "model__n_estimators": randint(100, 800),
        "model__min_samples_leaf": randint(1, 20),
        "model__max_features": uniform(0.2, 0.7),
    },
    "hist_gradient_boosting": {
        "model__max_iter": randint(200, 800),
        "model__learning_rate": loguniform(1e-2, 2e-1),
        "model__max_leaf_nodes": randint(15, 127),
        "model__min_samples_leaf": randint(5, 50),
        "model__l2_regularization": loguniform(1e-3, 1e1),
    },
    "xgboost": {
        "model__n_estimators": randint(200, 1000),
        "model__learning_rate": loguniform(1e-2, 2e-1),
        "model__max_depth": randint(3, 10),
        "model__subsample": uniform(0.6, 0.4),
        "model__colsample_bytree": uniform(0.6, 0.4),
    },
    "lightgbm": {
        "model__n_estimators": randint(200, 1000),
        "model__learning_rate": loguniform(1e-2, 2e-1),
        "model__num_leaves": randint(15, 127),
        "model__subsample": uniform(0.6, 0.4),
        "model__colsample_bytree": uniform(0.6, 0.4),
    },
    "catboost": {
        "model__iterations": randint(200, 800),
        "model__learning_rate": loguniform(1e-2, 2e-1),
        "model__depth": randint(3, 10),
        "model__subsample": uniform(0.6, 0.4),
    },
    "mlp": {
        "model__hidden_layer_sizes": [(64,), (128, 64), (64, 32), (256, 128), (128,)],
        "model__alpha": loguniform(1e-5, 1e-1),
        "model__learning_rate_init": loguniform(1e-4, 1e-2),
    },
    "knn": {
        "model__n_neighbors": randint(3, 100),
        "model__weights": ["uniform", "distance"],
        "model__p": [1, 2],
    },
}

inner_cv = GroupKFold(n_splits=CV_FOLDS)
groups_train = train.loc[has_target_train, GROUP]
search_rows = []

for architecture in ARCHITECTURES_TO_TUNE:
    model_name = f"{architecture}_tuned"
    scale, impute = PREPROCESSING_NEEDS[architecture]
    base = Pipeline([
        ("prepare", build_preprocessor(ACCEPTED_NUMERIC, CATEGORICAL,
                                       impute=impute, scale=scale)),
        ("model", ESTIMATOR_CLASSES[architecture](**FIXED_KWARGS.get(architecture, {}))),
    ])
    search = RandomizedSearchCV(
        base, PARAM_DISTRIBUTIONS[architecture],
        n_iter=N_SEARCH_ITERATIONS, cv=inner_cv,
        scoring="neg_mean_absolute_error", random_state=SEED,
        n_jobs=-1, refit=True,
    )
    started = time.perf_counter()
    search.fit(X_train.loc[has_target_train], y_train.loc[has_target_train],
               groups=groups_train)
    fitted_pipelines[model_name] = search.best_estimator_
    predictions[model_name] = search.predict(validation[ACCEPTED_FEATURES])
    assert np.isfinite(predictions[model_name].to_numpy(float)).all(), model_name
    search_rows.append({
        "model": model_name,
        "cv_mae_kt": -search.best_score_,
        "val_mae_kt": mean_absolute_error(y_validation, predictions[model_name]),
        "best hyperparameters": ", ".join(
            f"{k.removeprefix('model__')}={v}" for k, v in search.best_params_.items()),
    })
    print(f"tuned {model_name:32s} in {time.perf_counter() - started:6.1f} s")

note_validation_decision(
    f"randomized search per architecture "
    f"({N_SEARCH_ITERATIONS} candidates x {CV_FOLDS} grouped folds)",
    N_SEARCH_ITERATIONS * len(ARCHITECTURES_TO_TUNE), section="7")
display(pd.DataFrame(search_rows).round(3))

## 8. Results

One table, every model scored on exactly the same validation cases against the
same persistence reference. Models sharing a rank are tied at this evidence
level. Below it, the paired bootstrap against persistence, which is the stricter
test.

In [ ]:
delta_v_24_results = pd.concat([
    scoring_frame[[GROUP, "NAME", "BASIN", "SEASON", "ISO_TIME", "split", "USA_WIND"]]
        .rename(columns={"USA_WIND": "current_wind_kt"}),
    scoring_frame[TARGET].rename("observed_change_kt"),
    scoring_frame[[TARGET]],   # keep the raw TARGET column; score_all_models and the
                               # bootstrap helpers index frame[TARGET]
    scoring_frame[FUTURE_WIND].rename("observed_wind_kt"),
    predictions,
], axis=1)

MODELS_TO_RUN = list(predictions.columns)
for model_name in MODELS_TO_RUN:
    assert np.isfinite(delta_v_24_results[model_name].to_numpy(dtype=float)).all(), \
        f"{model_name} produced NaN/inf predictions"
metrics = score_all_models(delta_v_24_results, MODELS_TO_RUN)
ranked = rank_by_interval(metrics)
display(ranked[["rank", "cases", "storms", "mae", "mae_low", "mae_high",
                "rmse", "bias", "skill"]].round(3))

In [ ]:
plot_leaderboard(metrics, f"Everything fitted, {LEAD_HOURS}-hour lead")

print("paired storm-block deltas vs persistence (negative = model better):")
for model_name in MODELS_TO_RUN:
    if model_name == "persistence":
        continue
    delta, low, high = paired_storm_block_deltas(
        delta_v_24_results, delta_v_24_results[model_name].to_numpy(float),
        np.zeros(len(delta_v_24_results)))
    verdict = ("tied" if low <= 0 <= high
               else "resolvably better" if high < 0 else "resolvably worse")
    print(f"  {model_name:38s} {delta:+7.3f} [{low:+7.3f}, {high:+7.3f}]  {verdict}")

In [ ]:
# The final model: best validation MAE among everything fitted above.
best_model = str(metrics["mae"].idxmin())
best_low, best_high = float(metrics.loc[best_model, "mae_low"]), float(metrics.loc[best_model, "mae_high"])
print(f"final model: {best_model} - validation MAE "
      f"{metrics.loc[best_model, 'mae']:.3f} kt "
      f"(95% storm-block interval {best_low:.3f} to {best_high:.3f}), "
      f"skill {metrics.loc[best_model, 'skill']:+.3f} vs persistence "
      f"({metrics.loc['persistence', 'mae']:.2f} kt), "
      f"{int(metrics.loc[best_model, 'cases']):,} cases / "
      f"{int(metrics.loc[best_model, 'storms'])} storms")
note_validation_decision(f"selecting '{best_model}' as the final model", 1, section="8")

## 9. Per-storm evaluation

The pooled table weights each six-hourly case equally. Scoring storm by storm
gives a different picture: per-storm MAE, and per-storm nMAE (MAE divided by the
storm's mean intensity), so a 10 kt miss on a 40 kt storm is not treated the same
as on a 120 kt typhoon.

In [ ]:
def per_storm_scores(frame, model_names):
    records = []
    for sid, group in frame.groupby(GROUP):
        record = {"SID": sid, "NAME": group["NAME"].iloc[0],
                  "BASIN": group["BASIN"].iloc[0], "cases": len(group),
                  "mean_intensity_kt": group["current_wind_kt"].mean()}
        scale = record["mean_intensity_kt"]
        for model_name in model_names:
            model_mae = mean_absolute_error(group["observed_change_kt"], group[model_name])
            record[f"{model_name}_MAE"] = model_mae
            record[f"{model_name}_nMAE"] = model_mae / scale if scale > 0 else np.nan
        records.append(record)
    return pd.DataFrame(records)


storm_scores = per_storm_scores(delta_v_24_results, MODELS_TO_RUN)
print(f"scored {len(storm_scores)} storms individually")

for metric in ("MAE", "nMAE"):
    table = storm_scores[[f"{m}_{metric}" for m in MODELS_TO_RUN]].dropna()
    summary = pd.DataFrame({
        "median": table.median(),
        "IQR": table.quantile(0.75) - table.quantile(0.25),
        "p90": table.quantile(0.90),
    })
    summary.index = MODELS_TO_RUN
    print(f"\nper-storm {metric} distribution")
    display(summary.round(3 if metric == "nMAE" else 2))

figure = make_subplots(rows=1, cols=2,
                       subplot_titles=("Per-storm MAE (kt)", "Per-storm normalised MAE"))
for model_name in MODELS_TO_RUN:
    figure.add_box(y=storm_scores[f"{model_name}_MAE"].dropna(), name=model_name,
                   marker_color=PURPLE if not model_name in RULE_BASED_BASELINES else ORANGE,
                   row=1, col=1, showlegend=False)
    figure.add_box(y=storm_scores[f"{model_name}_nMAE"].dropna(), name=model_name,
                   marker_color=PURPLE if not model_name in RULE_BASED_BASELINES else ORANGE,
                   row=1, col=2, showlegend=False)
figure.update_layout(title=f"Per-storm errors at {LEAD_HOURS} h lead",
                     height=560, boxmode="group")
figure.update_xaxes(tickangle=-90)
figure.show()

## 10. Where does the final model fail?

MAE by basin, by season, and on rapid intensification / rapid weakening / ordinary
cases (±30 kt). Basin differences mix climate with sample composition, and
season-to-season jumps may be drift rather than model failure. The rapid-change
rows are where a squared-or-absolute loss on mostly ordinary cases tends to
underperform: the models usually beat persistence there, but less than they do
overall.

In [ ]:
RAPID_CHANGE_THRESHOLD_KT = 30

breakdown_rows = delta_v_24_results.copy()
breakdown_rows["case_group"] = np.select(
    [breakdown_rows["observed_change_kt"].ge(RAPID_CHANGE_THRESHOLD_KT),
     breakdown_rows["observed_change_kt"].le(-RAPID_CHANGE_THRESHOLD_KT)],
    [f"rapid intensification (+{RAPID_CHANGE_THRESHOLD_KT} kt or more)",
     f"rapid weakening (-{RAPID_CHANGE_THRESHOLD_KT} kt or less)"],
    default="ordinary change")


def group_scores(frame, group_columns):
    """MAE and MAE skill vs persistence, per model, within each group of rows."""
    records = []
    for group_keys, group in frame.groupby(group_columns, observed=True):
        group_keys = group_keys if isinstance(group_keys, tuple) else (group_keys,)
        record = dict(zip(group_columns, group_keys))
        persistence_mae = float(group["observed_change_kt"].abs().mean())
        record["cases"] = len(group)
        record["storms"] = group[GROUP].nunique()
        record["persistence_MAE"] = persistence_mae
        for model_name in MODELS_TO_RUN:
            model_mae = mean_absolute_error(group["observed_change_kt"], group[model_name])
            record[f"{model_name}_MAE"] = model_mae
            record[f"{model_name}_skill"] = 1.0 - model_mae / persistence_mae
        records.append(record)
    return pd.DataFrame(records)


spatial = group_scores(breakdown_rows, ["BASIN"])
temporal = group_scores(breakdown_rows, ["SEASON"])
anomaly = group_scores(breakdown_rows, ["case_group"])

print("by basin")
display(spatial.round(2))
print("by season")
display(temporal.round(2))
print(f"by case group (threshold {RAPID_CHANGE_THRESHOLD_KT} kt)")
display(anomaly.round(2))

In [ ]:
mae_columns = ["persistence_MAE"] + [f"{m}_MAE" for m in MODELS_TO_RUN]

figure = make_subplots(
    rows=1, cols=3,
    subplot_titles=("MAE by basin (persistence + first 5 models)",
                    f"{best_model}: skill vs persistence by basin",
                    "MAE by season (every model)"),
    specs=[[{"type": "xy"}, {"type": "xy"}, {"type": "xy"}]])

for i, column in enumerate(mae_columns[:6]):
    figure.add_bar(x=spatial["BASIN"], y=spatial[column], name=column,
                   marker_color=MODEL_COLORS[i], row=1, col=1, showlegend=False)

figure.add_bar(x=spatial["BASIN"], y=spatial[f"{best_model}_skill"],
               marker_color=PURPLE, showlegend=False, row=1, col=2)
figure.add_hline(y=0, line_color="black", row=1, col=2)

for model_name in MODELS_TO_RUN:
    figure.add_scatter(x=temporal["SEASON"], y=temporal[f"{model_name}_MAE"],
                       mode="lines+markers", name=model_name,
                       line=dict(color=ORANGE if model_name == "persistence" else PURPLE,
                                 width=1.5),
                       marker=dict(size=4), opacity=0.7,
                       row=1, col=3)

figure.update_layout(title=f"Failure breakdown, {LEAD_HOURS}-hour lead",
                     height=480, barmode="group",
                     legend=dict(font=dict(size=9)))
figure.update_yaxes(title_text="MAE (kt)", col=1)
figure.update_yaxes(title_text="skill", col=2)
figure.show()

figure = go.Figure()
for i, column in enumerate(mae_columns):
    name = "persistence" if column == "persistence_MAE" else column.removesuffix("_MAE")
    figure.add_bar(x=anomaly["case_group"], y=anomaly[column], name=name,
                   marker_color=ORANGE if name == "persistence" else PURPLE)
case_group_order = [f"rapid weakening (-{RAPID_CHANGE_THRESHOLD_KT} kt or less)",
                    "ordinary change",
                    f"rapid intensification (+{RAPID_CHANGE_THRESHOLD_KT} kt or more)"]
figure.update_layout(title=f"Ordinary vs rapid-change cases ({LEAD_HOURS} h lead)",
                     yaxis_title="MAE (kt)", barmode="group", height=480,
                     legend=dict(font=dict(size=9)))
figure.update_xaxes(categoryorder="array", categoryarray=case_group_order)
figure.show()

In [ ]:
# Where the final model is most wrong, and what a forest was looking at.
random_forest_name = next((m for m in MODELS_TO_RUN if m.startswith("random_forest")), None)
random_forest_pipeline = fitted_pipelines[random_forest_name]
transformed_names = random_forest_pipeline.named_steps["prepare"].get_feature_names_out()
importances = pd.Series(
    random_forest_pipeline.named_steps["model"].feature_importances_,
    index=transformed_names).nlargest(15).sort_values()

figure = go.Figure(go.Bar(
    x=importances.to_numpy(), y=list(importances.index), orientation="h",
    marker_color=PURPLE,
    hovertemplate="%{y}: %{x:.3f}<extra></extra>"))
figure.update_layout(title=f"Random forest ({random_forest_name}): top 15 importances",
                     xaxis_title="impurity-based importance", height=480,
                     showlegend=False)
figure.show()

worst_misses = delta_v_24_results.assign(
    prediction_error_kt=lambda frame: frame["observed_change_kt"] - frame[best_model],
).nlargest(10, "prediction_error_kt")
display(worst_misses[[GROUP, "NAME", "BASIN", "ISO_TIME", "current_wind_kt",
                      "observed_change_kt", best_model,
                      "prediction_error_kt"]].round(1))

## 11. Takeaways

(Filled in after the run - see the executed numbers above.)

- **Read the rank column, not the MAE column.** Overlapping storm-block intervals
  mean tied, however confident the point ranking looks.
- **Quote against persistence, with counts.** Climatology by basin x intensity is
  the bar: improvements beyond it are what the predictors actually buy.
- **Variants vs tuned.** Each `_v1`/`_v2` pair is a hand-picked contrast; each
  `_tuned` model was selected by grouped CV on the training split only. The gap
  between a tuned model's CV MAE and its validation MAE is partly selection
  optimism - the ledger counts every consultation.
- **The Gaussian process** was fitted on 2,000 training rows and the directional
  model predicts only the sign, so neither is directly comparable to the rest.
- **Rapid-change cases** are where the remaining error lives; the pooled MAE
  understates it.

## 12. Run record and final model export

In [ ]:
decision_ledger = pd.DataFrame(VALIDATION_DECISIONS)
display(decision_ledger)
print(f"validation-set decisions: {int(decision_ledger['candidates'].sum())}")

run_record = {
    "dataset": dataset_path.name,
    "target": TARGET,
    "seed": SEED,
    "software": {
        "python": sys.version.split()[0],
        "numpy": np.__version__, "pandas": pd.__version__,
        "scikit-learn": sklearn.__version__,
    },
    "feature_blocks": BLOCKS,
    "excluded_columns": EXCLUDED_COLUMNS,
    "models": MODELS_TO_RUN,
    "final_model": best_model,
    "final_model_validation_mae_kt": float(metrics.loc[best_model, "mae"]),
    "tuning": {"search": "RandomizedSearchCV", "cv": "GroupKFold on SID",
               "folds": CV_FOLDS, "candidates_per_architecture": N_SEARCH_ITERATIONS},
    "test_period_touched": False,
    "validation_decisions": VALIDATION_DECISIONS,
}

metrics.round(4).to_csv(OUTPUT_DIRECTORY / f"sklearn_baselines_leaderboard_{LEAD_HOURS}h.csv")
decision_ledger.to_csv(OUTPUT_DIRECTORY / "sklearn_baselines_validation_decisions.csv",
                       index=False)
(OUTPUT_DIRECTORY / "sklearn_baselines_run_record.json").write_text(
    json.dumps(run_record, indent=2, default=str) + "\n")

# Export the final fitted pipeline for Thursday's notebook.
final_artifact = {
    "model_name": best_model,
    "lead_hours": LEAD_HOURS,
    "target_column": TARGET,
    "feature_columns": ACCEPTED_FEATURES,
    "pipeline": fitted_pipelines[best_model],
    "training_cases": int(has_target_train.sum()),
    "training_storms": int(train.loc[has_target_train, GROUP].nunique()),
}
artifact_path = MODEL_DIRECTORY / f"sklearn_baselines_final_{LEAD_HOURS}h.joblib"
import joblib
joblib.dump(final_artifact, artifact_path)

# round-trip check
reloaded = joblib.load(artifact_path)
check = train.loc[has_target_train, ACCEPTED_FEATURES].head(5)
np.testing.assert_allclose(reloaded["pipeline"].predict(check),
                           fitted_pipelines[best_model].predict(check))
print(f"written to {OUTPUT_DIRECTORY}")
print(f"final pipeline saved to {artifact_path} (round-trip check passed)")